In [1]:
import os
import numpy as np
import pandas as pd
import time

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from PIL import Image
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}  |  GPUs: {torch.cuda.device_count()}")

Device: cuda  |  GPUs: 2


In [2]:
import kagglehub
path = kagglehub.dataset_download("piyshsss/acne04")
classification_path = os.path.join(path, "Classification", "Classification")
images_dir = os.path.join(classification_path, "JPEGImages")
print("Path:", path)

Path: /kaggle/input/datasets/piyshsss/acne04


In [3]:
def parse_label_file(filepath):
    rows = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            fname, level, count = line.split()
            rows.append((fname, int(level), int(count)))
    return pd.DataFrame(rows, columns=["filename", "level", "count"])

def map_severity(count):
    if 1 <= count <= 5:
        return "mild"
    elif 6 <= count <= 20:
        return "moderate"
    else:
        return "severe"

train_df = parse_label_file(os.path.join(classification_path, "NNEW_trainval_0.txt"))
test_df  = parse_label_file(os.path.join(classification_path, "NNEW_test_0.txt"))
for df in (train_df, test_df):
    df["severity"] = df["count"].apply(map_severity)

print(f"Train: {len(train_df)}  Test: {len(test_df)}  Total: {len(train_df)+len(test_df)}")
print(train_df["severity"].value_counts())

Train: 1165  Test: 292  Total: 1457
severity
moderate    506
mild        410
severe      249
Name: count, dtype: int64


In [4]:
train_core_df, val_df = train_test_split(
    train_df, test_size=0.15, stratify=train_df["severity"], random_state=42
)
train_core_df = train_core_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"train_core: {len(train_core_df)}  val: {len(val_df)}  test: {len(test_df)}")
print("\ntrain_core severity distribution (real, unresampled):")
print(train_core_df["severity"].value_counts())

train_core: 990  val: 175  test: 292

train_core severity distribution (real, unresampled):
severity
moderate    430
mild        348
severe      212
Name: count, dtype: int64


In [5]:
SEVERITY_TO_IDX = {"mild": 0, "moderate": 1, "severe": 2}

# Balanced class weights: N_total / (num_classes * N_c)
counts = train_core_df["severity"].value_counts()
n_total = len(train_core_df)
class_weights = torch.tensor(
    [n_total / (3 * counts[c]) for c in ["mild", "moderate", "severe"]],
    dtype=torch.float32
).to(device)
print("Class weights [mild, moderate, severe]:", class_weights.tolist())

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomPerspective(distortion_scale=0.15, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.2),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class AcneSeverityDataset(Dataset):
    def __init__(self, df, images_dir, transform):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.images_dir, row["filename"])).convert("RGB")
        image = self.transform(image)
        return {
            "image": image,
            "severity_idx": torch.tensor(SEVERITY_TO_IDX[row["severity"]], dtype=torch.long),
        }

train_dataset = AcneSeverityDataset(train_core_df, images_dir, train_transform)
val_dataset   = AcneSeverityDataset(val_df, images_dir, eval_transform)
test_dataset  = AcneSeverityDataset(test_df, images_dir, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"train: {len(train_dataset)}  val: {len(val_dataset)}  test: {len(test_dataset)}")

Class weights [mild, moderate, severe]: [0.9482758641242981, 0.7674418687820435, 1.5566037893295288]
train: 990  val: 175  test: 292


In [6]:
class AcneSeverityResNet50(nn.Module):
    def __init__(self, num_classes=3, dropout=0.3):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        feat_dim = backbone.fc.in_features

        for name, module in self.features.named_children():
            if name in ["0", "1", "4"]:  # conv1, bn1, layer1 frozen; layer2+ trainable
                for p in module.parameters():
                    p.requires_grad = False

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        feats = self.features(x).flatten(1)
        feats = self.dropout(feats)
        return self.fc(feats)

model = AcneSeverityResNet50().to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 177MB/s]


Trainable params: 23,288,835 / 23,514,179


In [7]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.SGD(filter(lambda p: p.requires_grad, model.parameters()),
                             lr=0.001, momentum=0.9, weight_decay=5e-4)

In [8]:
model.train()
batch = next(iter(train_loader))
images = batch["image"].to(device)
labels = batch["severity_idx"].to(device)

optimizer.zero_grad()
logits = model(images)
loss = criterion(logits, labels)
loss.backward()
optimizer.step()

print(f"logits shape: {logits.shape}  loss: {loss.item():.4f}")

logits shape: torch.Size([32, 3])  loss: 1.1270


In [9]:
def evaluate(model, loader):
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device)
            labels = batch["severity_idx"]
            logits = model(images)
            preds = logits.argmax(dim=1).cpu()
            all_true.extend(labels.tolist())
            all_pred.extend(preds.tolist())

    acc = sum(t == p for t, p in zip(all_true, all_pred)) / len(all_true)
    cm = np.zeros((3, 3), dtype=int)
    for t, p in zip(all_true, all_pred):
        cm[t][p] += 1
    return acc, cm

In [10]:
def get_module(m):
    return m.module if isinstance(m, nn.DataParallel) else m

PATIENCE = 15
NUM_EPOCHS = 120
best_val_acc = 0.0
patience_counter = 0
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    for pg in optimizer.param_groups:
        pg["lr"] = 0.001 * (0.5 ** ((epoch - 1) // 30))

    model.train()
    running_loss = 0.0
    for batch in train_loader:
        images = batch["image"].to(device)
        labels = batch["severity_idx"].to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)
    val_acc, _ = evaluate(get_module(model), val_loader)
    elapsed = (time.time() - start_time) / 60

    print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | train_loss={epoch_loss:.4f} | "
          f"val_acc={val_acc:.4f} | lr={optimizer.param_groups[0]['lr']:.5f} | "
          f"elapsed={elapsed:.1f}min", flush=True)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(get_module(model).state_dict(), "/kaggle/working/best_model_exp2.pt")
        patience_counter = 0
        print(f"  -> new best (val_acc={best_val_acc:.4f}), checkpoint saved", flush=True)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}", flush=True)
            break

print(f"\nTraining finished. Best val accuracy: {best_val_acc:.4f}")

Epoch   1/120 | train_loss=1.0776 | val_acc=0.4629 | lr=0.00100 | elapsed=1.5min
  -> new best (val_acc=0.4629), checkpoint saved
Epoch   2/120 | train_loss=1.0167 | val_acc=0.4914 | lr=0.00100 | elapsed=2.9min
  -> new best (val_acc=0.4914), checkpoint saved
Epoch   3/120 | train_loss=0.9811 | val_acc=0.4914 | lr=0.00100 | elapsed=4.3min
Epoch   4/120 | train_loss=0.9374 | val_acc=0.5029 | lr=0.00100 | elapsed=5.8min
  -> new best (val_acc=0.5029), checkpoint saved
Epoch   5/120 | train_loss=0.8977 | val_acc=0.5200 | lr=0.00100 | elapsed=7.2min
  -> new best (val_acc=0.5200), checkpoint saved
Epoch   6/120 | train_loss=0.8700 | val_acc=0.6171 | lr=0.00100 | elapsed=8.6min
  -> new best (val_acc=0.6171), checkpoint saved
Epoch   7/120 | train_loss=0.8560 | val_acc=0.6229 | lr=0.00100 | elapsed=10.0min
  -> new best (val_acc=0.6229), checkpoint saved
Epoch   8/120 | train_loss=0.8103 | val_acc=0.6514 | lr=0.00100 | elapsed=11.3min
  -> new best (val_acc=0.6514), checkpoint saved
Epoch  

In [11]:
final_model = AcneSeverityResNet50().to(device)
final_model.load_state_dict(torch.load("/kaggle/working/best_model_exp2.pt"))

test_acc, cm = evaluate(final_model, test_loader)
print(f"FINAL TEST SEVERITY ACCURACY: {test_acc:.4f}")
print("\nConfusion matrix (rows=true, cols=pred) [mild, moderate, severe]:")
print(cm)

for i, name in enumerate(["mild", "moderate", "severe"]):
    recall = cm[i][i] / cm[i].sum()
    print(f"{name} recall: {recall:.4f}")

FINAL TEST SEVERITY ACCURACY: 0.8322

Confusion matrix (rows=true, cols=pred) [mild, moderate, severe]:
[[ 85  18   0]
 [ 10 105  12]
 [  0   9  53]]
mild recall: 0.8252
moderate recall: 0.8268
severe recall: 0.8548
